# COMP41810 Advanced Machine Learning

- **Name:** 
- **Student Number:**
- **Name:** 
- **Student Number:**
- **Name:** 
- **Student Number:** 

## Assignment 2: Learning to Walk *With Style*
Uses the PPO actor-critic method to train a neural network to allow a simple robot to learn to walk using the **Bipedal Walker** environment from the `gymnasium` package (https://gymnasium.farama.org/environments/box2d/bipedal_walker/). This assignment uses a custom version of the environment that includes some small modifications, `BipedalWalkerUCD`.

![Bipedal Walker](bipedal_walker.gif)

The **action** space is continuous where the actions are motor speed values in the [-1, 1] range for each of the 4 joints at both hips and knees.

The **observation** is a state vector consisting of hull angle speed, angular velocity, horizontal speed, vertical speed, position of joints and joints angular speed, legs contact with ground, and 10 lidar rangefinder measurements. There are no coordinates in the state vector.

The basic **reward** provides a small positive rewqard for every step forwards (left to right) the agent takes.

In this assignment we will explore different reward functions. 

### Initialisation

If using Google colab you need to install packages - comment out lines below.

In [ ]:
#!apt install swig cmake ffmpeg
#!apt-get install -y xvfb x11-utils
#!pip install stable-baselines3[extra] pyglet box2d box2d-kengz
#!pip install pyvirtualdisplay PyOpenGL PyOpenGL-accelerate
#!pip install gymnasium stable_baselines3

For Google colab comment out this cell to make a virtual rendering canvas so render calls work (we still wont; see display!)

In [ ]:
#import pyvirtualdisplay
#
#_display = pyvirtualdisplay.Display(visible=False,  # use False with Xvfb
#                                    size=(1400, 900))
#_ = _display.start()

Import required packages.

In [ ]:
import gymnasium as gym
import stable_baselines3 as sb3
import torch

import bipedal_walker_ucd

import os
import pandas as pd # For data frames and data frame manipulation
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 1000)
import numpy as np # For general  numeric operations
from collections import deque
from pathlib import Path
import re

import imageio
from IPython.display import Image as DisplayImage, Video, display
from PIL import Image, ImageDraw
import imageio.v3 as iio

import matplotlib.pyplot as plt
%matplotlib inline

### Define Utility Code

We define a custom wrapper to allow experimantion twith different rewards in the Bipedal Walker environment. This adds potential extra reward terms:

- **Action Penalty:** Encourages the agent tio be frugal in its use of actions.
- **Stability:** Encourage the hull to stay horizontal.
- **Smoothness:** Encourages similar action from one step to the next.
- **Speed:** Encourages the agent to move as fast as possible. 

Each of these different com-oentnts of the reward can have a different weight associated with it. These are passed as parameters to the reward wrapper.

- `action_penalty_weight`
- `stability_weight`
- `speed_weight`
- `smoothness_weight`


In [ ]:
class CustomRewardWrapper(gym.Wrapper):
    def __init__(self, env, window=20,
                 action_penalty_weight = 0.0,
                 stability_weight=0.0,
                 smoothness_weight = 0.0,
                 speed_weight=0.0
                 ):
        super().__init__(env)
        self.window = window
        self.action_penalty_weight = action_penalty_weight
        self.stability_weight = stability_weight
        self.smoothness_weight = smoothness_weight
        self.speed_weight = speed_weight

        self.prev_action = np.zeros(env.action_space.shape)
        self.hip1_history = deque(maxlen=window)
        self.hip2_history = deque(maxlen=window)

    def step(self, action):
        obs, base_reward, terminated, truncated, info = self.env.step(action)

        # --- Extract state ---
        hull = self.env.unwrapped.hull
        hull_angle   = hull.angle
        hull_vel_x   = hull.linearVelocity[0]
        hull_vel_y   = hull.linearVelocity[1]
        hull_height  = hull.position.y

        vel         = obs[2]
        hip1_angle  = obs[4]
        hip2_angle  = obs[10]
        hip1_vel    = obs[5]
        hip2_vel    = obs[11]
        knee1_angle = obs[6]
        knee2_angle = obs[12]
        knee1_vel   = obs[7]
        knee2_vel   = obs[13]

        # --- Action Penalty ----
        # normalized to about -50.0 using heuristic, more optimal agent should spend less
        action_penalty = 0
        for a in action:
            action_penalty -= 0.00035 * bipedal_walker_ucd.MOTORS_TORQUE * np.clip(np.abs(a), 0, 1)
        action_penalty = action_penalty

        # --- Stability ---
        stability = -abs(hull_angle)

        # --- Smoothness ---
        smoothness = -np.sum(np.abs(action - self.prev_action))

        # --- Speed targeting ---
        speed = hull_vel_x

        # --- Total reward ---
        modified_reward = (
            base_reward
            + action_penalty * self.action_penalty_weight
            + stability * self.stability_weight
            + smoothness * self.smoothness_weight
            + speed * self.speed_weight
        )

        # --- Reward components for TensorBoard ---
        info["reward_components"] = {
            "base":              base_reward,
            "action_penalty":    action_penalty,
            "stability":         stability,
            "smoothness":        smoothness,
            "speed":             speed,
            "total":             modified_reward,
        }

        self.prev_action = action.copy()
        return obs, modified_reward, terminated, truncated, info

    def reset(self, **kwargs):
        self.prev_action = np.zeros(self.env.action_space.shape)
        self.hip1_history.clear()
        self.hip2_history.clear()
        return self.env.reset(**kwargs)

The `VideoEvalCallback` writes out the components of the reward function defined above and also records videos of agent behaviour.

In [ ]:

class VideoEvalCallback(sb3.common.callbacks.BaseCallback):
    def __init__(
        self,
        eval_env: gym.Env,
        eval_freq: int = 10_000,
        n_eval_episodes: int = 1,
        video_folder: str = "./eval_videos",
        video_length: int = 1000,
        tensorboard_log: str = "./tb_logs",
        tb_video_every: int = 4,
        tb_video_scale: int = 2,
        verbose: int = 1,
    ):
        super().__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.n_eval_episodes = n_eval_episodes
        self.video_folder = video_folder
        self.video_length = video_length        # max frames recorded per evaluation video
        self.tensorboard_log = tensorboard_log
        self.tb_video_every = tb_video_every    # TensorBoard copy keeps every Nth frame (0 = no TensorBoard video)...
        self.tb_video_scale = tb_video_scale    # ...downscaled by this factor
        self.writer = None                      # opened on first use, see _get_writer()
        self.eval_count = 0
        os.makedirs(video_folder, exist_ok=True)

    def _get_writer(self):
        # Opened lazily so that creating a callback (or re-running its cell) doesn't leave a writer open
        if self.writer is None:
            self.writer = torch.utils.tensorboard.SummaryWriter(log_dir=self.tensorboard_log)
        return self.writer

    def _on_step(self) -> bool:
        if self.n_calls % self.eval_freq == 0:
            self._run_eval()
        return True

    def _run_eval(self):
        self.eval_count += 1
        episode_rewards = []
        component_totals = {}
        tb_frames = []  # small copies of some frames, for TensorBoard
        video_path = os.path.join(
            self.video_folder, f"eval_{self.eval_count}_step_{self.num_timesteps}.mp4"
        )

        for episode in range(self.n_eval_episodes):
            obs, _ = self.eval_env.reset()
            done = False
            total_reward = 0.0
            ep_components = {}
            # Episode 0 is recorded: frames are written straight to the mp4 rather than kept in memory
            video_writer = imageio.get_writer(video_path, fps=30) if episode == 0 else None
            n_frames = 0

            try:
                while not done:
                    if video_writer is not None and n_frames < self.video_length:
                        frame = self.eval_env.render()
                        if frame is not None:
                            video_writer.append_data(frame)
                            if self.tb_video_every and n_frames % self.tb_video_every == 0:
                                # render() returns a view into the whole scrolled canvas;
                                # .copy() keeps only this small frame, not the canvas
                                s = self.tb_video_scale
                                tb_frames.append(frame[::s, ::s].copy())
                            n_frames += 1

                    action, _ = self.model.predict(obs, deterministic=True)
                    obs, reward, terminated, truncated, info = self.eval_env.step(action)
                    total_reward += reward
                    done = terminated or truncated

                    components = info.get("reward_components", {})
                    for key, val in components.items():
                        ep_components[key] = ep_components.get(key, 0.0) + val
            finally:
                if video_writer is not None:
                    video_writer.close()  # also stops the ffmpeg process if training is interrupted

            episode_rewards.append(total_reward)

            for key, val in ep_components.items():
                component_totals[key] = component_totals.get(key, 0.0) + val

        mean_reward = np.mean(episode_rewards)
        std_reward  = np.std(episode_rewards)

        writer = self._get_writer()
        writer.add_scalar("eval/mean_reward", mean_reward, self.num_timesteps)
        writer.add_scalar("eval/std_reward",  std_reward,  self.num_timesteps)

        for key, total in component_totals.items():
            mean_val = total / self.n_eval_episodes
            writer.add_scalar(
                f"eval/reward_components/{key}", mean_val, self.num_timesteps
            )

        if self.verbose:
            print(f"\n[Eval @ {self.num_timesteps} steps]")
            print(f"  Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")
            for key, total in component_totals.items():
                print(f"    {key:>20s}: {total / self.n_eval_episodes:.4f}")
            print(f"  Video saved to {video_path}")

        if tb_frames:
            self._log_video_to_tensorboard(tb_frames)

    def _log_video_to_tensorboard(self, frames: list):
        # (1, T, C, H, W) uint8 -- add_video rescales uint8 itself, so no float32 copy is needed here
        video_array = np.stack(frames).transpose(0, 3, 1, 2)[np.newaxis]
        writer = self._get_writer()
        writer.add_video(
            "eval/agent_behaviour",
            video_array,
            global_step=self.num_timesteps,
            fps=max(1, round(30 / self.tb_video_every)),
        )
        writer.flush()

    def close(self):
        """Close the TensorBoard writer. Done automatically when training finishes normally;
        call it yourself after a manual evaluation or an interrupted learn()."""
        if self.writer is not None:
            self.writer.close()
            self.writer = None

    def _on_training_end(self):
        self.close()

A function to create visual summaries of videos saved for easy viewing. (Thanks to Gareth Morgan for this function)

In [ ]:
def create_quick_frame_summary(video_path, n_samples=10):
    video_path = Path(video_path)
    output_dir = Path(video_path.parent / "frame_summaries")
    
    output_dir.mkdir(parents=True, exist_ok=True)

    frames = iio.imread(video_path)
    n_frames = len(frames)
    picks = sorted(
        set(int(round(i * (n_frames - 1) / max(n_samples - 1, 1))) for i in range(n_samples))
    )

    selected = [Image.fromarray(frames[i]) for i in picks]
    width, height = selected[0].size
    label_height = 28
    canvas = Image.new("RGB", (width * len(selected), height + label_height), color=(20, 20, 20))
    draw = ImageDraw.Draw(canvas)

    for idx, (frame_idx, frame_img) in enumerate(zip(picks, selected)):
        x = idx * width
        canvas.paste(frame_img, (x, label_height))
        draw.text((x + 10, 6), f"frame {frame_idx}", fill=(255, 255, 255))

    summary_path = output_dir / f"{video_path.stem}_summary.jpg"
    canvas.save(summary_path, quality=90)
    return summary_path


def select_evenly_spaced(folder, n_select=5, pattern=r"eval_(\d+)_step_(\d+)\.mp4$"):
   
    """Return n_select video paths evenly spaced by index, including first and last."""
    regex = re.compile(pattern)
    videos = []
    for p in Path(folder).iterdir():
        m = regex.match(p.name)
        if m:
            videos.append((int(m.group(1)), int(m.group(2)), p))

    # Sort numerically by index X (not alphabetically)
    videos.sort(key=lambda v: v[0])

    n = len(videos)
    if n == 0:
        return []
    if n <= n_select:
        return [v[2] for v in videos]

    # Evenly spaced positions from 0 to n-1 inclusive
    positions = [round(i * (n - 1) / (n_select - 1)) for i in range(n_select)]
    positions = sorted(set(positions))  # guard against duplicates
    return [videos[i][2] for i in positions]


### Task 1 - Run A Custom Reward Wegith Experiment
Create an instance of the **BipedalWalkerUCD** environment and add a `CustomRewardWrapper` defining the reward mix that will be used in the experiment. Experiment with using different weights in the reward function.

We define a set of utility values used throughout the experiment.

In [ ]:
time_limit = 5000 # A time out for episodes (avoids infinte runs)
eval_freq = 5000 # Frequency with which evlaution (and generation of evaluation vidoe) is executed
n_eval_episodes = 3 # Number of evalaution episodes across which scores are generated
tb_log_path = './log_tb_bipedal/'
eval_log_path = './logs_bipedal/'
eval_video_base_path = Path('./eval_videos/')
run_name = "Reward_Exploration"
eval_video_path = eval_video_base_path / run_name
tb_log_name = run_name

#### Create the Training Environment
Experiment with different weights for different compoennts of the reward function. 

In [ ]:
action_penalty_weight = # Add code here
stability_weight = # Add code here
smoothness_weight = # Add code here
speed_weight = # Add code here

Create the environment.

In [ ]:
env = bipedal_walker_ucd.BipedalWalkerUCD(render_mode="rgb_array")
env = CustomRewardWrapper(env,
                          action_penalty_weight = action_penalty_weight,
                          stability_weight = stability_weight,
                          smoothness_weight = smoothness_weight,
                          speed_weight = speed_weight
                         )
env = gym.wrappers.TimeLimit(env,
                                max_episode_steps = time_limit)

#### Create an Agent
Create an agent that controls the robot. A `PPO` agent with the following hyper-parameters should work well (although you can experiment):
- `batch_size = 64`
- `clip_range = 0.2`
- `ent_coef = 0.001`
- `gae_lambda =  0.95`
- `gamma = 0.99`
- `learning_rate = 0.00025`
- `n_steps = 2048`
- `n_epochs = 10`

We also recommend enabling **tensorboard** monitoring of the training process.

In [ ]:
agent = # Add code here


Examine the actor and critic network architectures.

In [ ]:
print(agent.policy)

#### Create a Callback
Create an evaluation callback that is called every at regular intervals and renders the episode. First create an evaluation environment with equivalent wrappers then create the callbacks.

In [ ]:
eval_env = bipedal_walker_ucd.BipedalWalkerUCD(render_mode="rgb_array")
eval_env = CustomRewardWrapper(eval_env,
                          action_penalty_weight = action_penalty_weight,
                          stability_weight = stability_weight,
                          smoothness_weight = smoothness_weight,
                          speed_weight = speed_weight
                         )
eval_env = gym.wrappers.TimeLimit(eval_env,
                                max_episode_steps = time_limit)

eval_log_path = eval_log_path

video_callback = VideoEvalCallback(
    eval_env = eval_env,
    eval_freq = eval_freq,       
    n_eval_episodes = n_eval_episodes,      
    video_folder = eval_video_path,
    tensorboard_log = tb_log_path,
    verbose=1,
)

#### Run the Agent Training Process
Train the model for a large number of timesteps (500,000 timesteps will probably work well).

In [ ]:
# Add code here


Connect to the tensorboard log using **TensorBoard** from the command line to view training progress:

    `tensorboard --logdir ./log_tb_bipedal/` # Change to whatever is stored in tb_log_path

Then open TensorBoard in a browser, typically located at:

`http://localhost:6006/`

#### View Video Summaries

Visualise some summaries of saved videos.

In [ ]:
for video_path in select_evenly_spaced(eval_video_path):
    
    print(f"Processing {video_path.name}")
        
    display(Video(str(video_path), embed=True, width=300))
    
    summary_path = create_quick_frame_summary(video_path)
    display(DisplayImage(filename=str(summary_path)))

#### Save the Agent
Save the trained agent.

In [ ]:
agent.save("./agents/" + run_name)

For memory management delete old agent and environment (assumes variable names - change if required).

In [ ]:
agent.logger.close()     
video_callback.close()   
env.close()
eval_env.close()
del agent, env, eval_env, video_callback

gc.collect()             

#### Reflection

*Add a reflection on the performance of this agent (<350 words)*

*Add reflection here*

### Task 2 - Defining a Custom Reward Component

Define new version of the `CustomRewardWrapper` that introduces a new compoent into the reward function. The parts of the state extracted are likely to be useful for this.

#### Create a New Custom Reward Wrapper

Add a new term to this new custom reward function.

In [ ]:
class CustomRewardWrapper(gym.Wrapper):
    def __init__(self, env, window=20,
                 action_penalty_weight = 1.0,
                 symmetry_weight=0.2,
                 stability_weight=0.3,
                 smoothness_weight = 0.2,
                 speed_weight=0.5,
                 hop_height_weight=1.0,
                 hop_velocity_weight=0.5,
                 base_height=1.4):
        super().__init__(env)
        self.window = window
        self.action_penalty_weight = action_penalty_weight
        self.stability_weight = stability_weight
        self.smoothness_weight = smoothness_weight
        self.speed_weight = speed_weight
        self.custom_component_weight = custom_component_weight # Add custom component weight here

        self.prev_action = np.zeros(env.action_space.shape)
        self.hip1_history = deque(maxlen=window)
        self.hip2_history = deque(maxlen=window)

    def step(self, action):
        obs, base_reward, terminated, truncated, info = self.env.step(action)

        # --- Extract state ---
        hull = self.env.unwrapped.hull
        hull_angle   = hull.angle
        hull_vel_x   = hull.linearVelocity[0]
        hull_vel_y   = hull.linearVelocity[1]
        hull_height  = hull.position.y

        hip1_angle  = obs[4]
        hip2_angle  = obs[10]
        hip1_vel    = obs[5]
        hip2_vel    = obs[11]
        knee1_angle = obs[6]
        knee2_angle = obs[12]
        knee1_vel   = obs[7]
        knee2_vel   = obs[13]

        # --- Action Penalty ----
        # normalized to about -50.0 using heuristic, more optimal agent should spend less
        action_penalty = 0
        for a in action:
            action_penalty -= 0.00035 * bipedal_walker_ucd.MOTORS_TORQUE * np.clip(np.abs(a), 0, 1)
        action_penalty = action_penalty

        # --- Stability ---
        stability = -abs(hull_angle)

        # --- Smoothness ---
        smoothness = -np.sum(np.abs(action - self.prev_action))

        # --- Speed targeting ---
        speed = hull_vel_x


        # --- Custom Component ---
        custom_component_reward = 0.0 # Add custom component logic here 

        # --- Total reward ---
        modified_reward = (
            base_reward
            + action_penalty * self.action_penalty_weight
            + stability * self.stability_weight
            + smoothness * self.smoothness_weight
            + speed * self.speed_weight
            + custom_component_reward * self.custom_component_weight  # Add custom component here
                
        # --- Reward components for TensorBoard ---
        info["reward_components"] = {
            "base":              base_reward,
            "action_penalty":    action_penalty,
            "stability":         stability,
            "smoothness":        smoothness,
            "speed":             speed,
            "custom_component_reward":    custom_component_reward,            
            "total":             modified_reward,
        }

        self.prev_action = action.copy()
        return obs, modified_reward, terminated, truncated, info

    def reset(self, **kwargs):
        self.prev_action = np.zeros(self.env.action_space.shape)
        self.hip1_history.clear()
        self.hip2_history.clear()
        return self.env.reset(**kwargs)

#### Run A Custom Experiment
Create an instance of the **BipedalWalkerUCD** environment and add a `CustomRewardWrapper` defining the reward mix that will be used in the experiment.

In [ ]:
action_penalty_weight = # Add code here
stability_weight = # Add code here
speed_weight = # Add code here
smoothness_weight = # Add code here
custom_component_weight = # Add code here

We also define a set of utility values used throughout the experiment.

In [ ]:
run_name = "Custom_Reward"
eval_video_path = eval_video_base_path / run_name
tb_log_name = run_name

Create the environment.

In [ ]:
env = bipedal_walker_ucd.BipedalWalkerUCD(render_mode="rgb_array")
env = CustomRewardWrapper(env,
                          action_penalty_weight = action_penalty_weight,
                          symmetry_weight = symmetry_weight,
                          stability_weight = stability_weight,
                          smoothness_weight = smoothness_weight,
                          speed_weight = speed_weight,
                          hop_height_weight = hop_height_weight,
                          hop_velocity_weight = hop_velocity_weight
                         )
env = gym.wrappers.TimeLimit(env,
                                max_episode_steps = time_limit)

Explore the environment - view the action space and observation space.

In [ ]:
env.action_space

In [ ]:
env.observation_space

#### Create an Agent
Create an agent that controls the robot. A `PPO` agent with the following hyper-parameters should work well (although you can experiment):
- `batch_size = 64`
- `clip_range = 0.2`
- `ent_coef = 0.001`
- `gae_lambda =  0.95`
- `gamma = 0.99`
- `learning_rate = 0.00025`
- `n_steps = 2048`
- `n_epochs = 10`

We also recommend enabling **tensorboard** monitoring of the training process.

In [ ]:
agent = # Add code here

Examine the actor and critic network architectures.

In [ ]:
print(agent.policy)

#### Create a Callback
Create an evaluation callback that is called every at regular intervals and renders the episode. First create an evaluation environment with equivalent wrappers then create the callbacks.

In [ ]:
eval_env = bipedal_walker_ucd.BipedalWalkerUCD(render_mode="rgb_array")
eval_env = CustomRewardWrapper(eval_env,
                          action_penalty_weight = action_penalty_weight,
                          symmetry_weight = symmetry_weight,
                          stability_weight = stability_weight,
                          smoothness_weight = smoothness_weight,
                          speed_weight = speed_weight,
                          hop_height_weight = hop_height_weight,
                          hop_velocity_weight = hop_velocity_weight
                         )
eval_env = gym.wrappers.TimeLimit(eval_env,
                                max_episode_steps = time_limit)

eval_log_path = eval_log_path

video_callback = VideoEvalCallback(
    eval_env = eval_env,
    eval_freq = eval_freq,        # evaluate every 20k steps
    n_eval_episodes = n_eval_episodes,       # average reward over 3 episodes, video from first
    video_folder = eval_video_path,
    tensorboard_log = tb_log_path,
    verbose=1,
)

#### Run the Agent Training Process
Train the model for a large number of timesteps (1,000,000 timesteps will probably work well).

In [ ]:
# Add code here


Connect to the tensorboard log using **TensorBoard** from the command line to view training progress:

    `tensorboard --logdir ./log_tb_bipedal/` # Change to whatever is stored in tb_log_path

Then open TensorBoard in a browser, typically located at:

`http://localhost:6006/`

#### View Video Summaries

Visualise some summaries of saved videos.

In [ ]:
for video_path in select_evenly_spaced(eval_video_path):
    
    print(f"Processing {video_path.name}")
        
    display(Video(str(video_path), embed=True, width=300))
    
    summary_path = create_quick_frame_summary(video_path)
    display(DisplayImage(filename=str(summary_path)))

#### Save the Agent
Save the trained agent.

In [ ]:
agent.save("./agents/" + run_name)

For memory management delete old agent and environment (assumes variable names - change if required).

In [ ]:
agent.logger.close()    
video_callback.close()  
env.close()
eval_env.close()
del agent, env, eval_env, video_callback

gc.collect()             # the callback and its stored locals form a reference cycle

#### Reflection

*Add a reflection on the performance of this agent (<350 words)*

*Add reflection here*

### Task 3 - Overall Reflection
*Write a short (no more than 500 words) reflection on the different agent you have trained, the performance of these agents, and how you might reflect on which one was best.*

*Add reflection here*